### Section 1: What are Guardrails?
Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

- Before the agent starts (input guardrails)
- After it completes (output guardrails)
- Around model and tool calls
Common Use Cases:
Use Case	Example
PII leakage prevention	Redact emails/credit cards before logging
Prompt injection blocking	Detect adversarial inputs
Harmful content filtering	Block dangerous requests
Business rule enforcement	Require approval for financial ops
Output quality validation	Ensure response meets safety standards


⚖️ Section 2: Two Approaches to Guardrails
Deterministic Guardrails
Rule-based: regex, keyword matching, explicit checks
- Fast, predictable, cost-effective
- May miss nuanced violations


Model-Based Guardrails
Uses LLMs/classifiers for semantic understanding
- Catches subtle/nuanced issues
- Slower and more expensive

In [5]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into database",
    "What is the capital of France",
    "Explain how malware spreads"
]

print("=== Deterministic Guardrail Demo ===")

for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
Blocked: How do I hack into database
Allowed: What is the capital of France
Blocked: Explain how malware spreads


In [8]:
from langchain_groq import ChatGroq

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety."""

    model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

    prompt = f"""
Is the following user input safe to process?

Reply with only 'Safe' or 'Unsafe'.

Input: {text}
"""

    result = model.invoke(prompt)
    return result.content.strip()


test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads."
]

print("=== Model-based Guardrail Demo ===")

for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if verdict.lower() == "unsafe" else "SAFE"
    print(f"{status}: {inp}")

=== Model-based Guardrail Demo ===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads.
